<a href="https://colab.research.google.com/github/ruvais-p/Auth-using-Clean-Architecture/blob/main/NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import re

# Load the dataset
try:
    df = pd.read_csv("/content/test_data.csv", encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv("/content/test_data.csv", encoding='latin-1')

# Drop rows with missing data
df = df.dropna(subset=['message', 'amount'])

# Clean the amount field
def clean_numeric(col):
    return (
        col.astype(str)
        .str.replace(r'[^0-9.]', '', regex=True)
        .str.extract(r'([\d.]+)', expand=False)
        .astype(float)
    )

df['amount'] = clean_numeric(df['amount'])

# Split features and target
X = df['message'].astype(str)
y = df['amount']

# Text vectorization
vectorizer = TfidfVectorizer(max_features=3000)
X_vec = vectorizer.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

# Train a regression model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"✅ MAE: {mae:.2f}")
print(f"✅ R² Score: {r2:.2f}")

# Save model and vectorizer
joblib.dump(model, "amount_regressor.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


✅ MAE: 3083.05
✅ R² Score: 0.60


['tfidf_vectorizer.pkl']

In [ ]:
# Load model
model = joblib.load("amount_regressor.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

# Example message
msg = "Dear UPI user A/C X5540 debited by 1080.0 on date 30Mar25 trf to MALAPPURAM TRADE Refno 016207021269. If not u? call 1800111109.�-SBI"

# Predict
X_input = vectorizer.transform([msg])
predicted_amount = model.predict(X_input)[0]
print(f"💰 Predicted amount: ₹{predicted_amount:.2f}")


💰 Predicted amount: ₹750.88


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("Dear UPI user A/C X5540 debited by 1080.0 on date 30Mar25 trf to MALAPPURAM TRADE Refno 016207021269. If not u? call 1800111109.�-SBI")

for token in doc:
    print(token.text, token.lemma_, token.pos_, token.tag_, token.dep_,
            token.shape_, token.is_alpha, token.is_stop)

Dear dear ADJ JJ compound Xxxx True False
UPI UPI PROPN NNP compound XXX True False
user user NOUN NN compound xxxx True False
A A PROPN NNP nmod X True True
/ / SYM SYM punct / False False
C c NOUN NN compound X True False
X5540 X5540 NOUN NNS ROOT Xdddd False False
debited debit VERB VBN acl xxxx True False
by by ADP IN agent xx True True
1080.0 1080.0 NUM CD pobj dddd.d False False
on on ADP IN prep xx True True
date date NOUN NN pobj xxxx True False
30Mar25 30mar25 NUM CD nummod ddXxxdd False False
trf trf PROPN NNP npadvmod xxx True False
to to ADP IN prep xx True True
MALAPPURAM MALAPPURAM PROPN NNP compound XXXX True False
TRADE TRADE PROPN NNP compound XXXX True False
Refno Refno PROPN NNP pobj Xxxxx True False
016207021269 016207021269 NUM CD nummod dddd False False
. . PUNCT . punct . False False
If if SCONJ IN mark Xx True True
not not PART RB neg xxx True True
u u NOUN NN nsubj x True False
? ? NOUN NN nsubj ? False False
call call VERB VB ROOT xxxx True True
1800111109. 18

In [ ]:

# English pipelines include a rule-based lemmatizer
nlp = spacy.load("en_core_web_sm")
lemmatizer = nlp.get_pipe("lemmatizer")
print(lemmatizer.mode)  # 'rule'
print([token.lemma_ for token in doc])
# ['I', 'be', 'read', 'the', 'paper', '.']


rule
['dear', 'UPI', 'user', 'A', '/', 'c', 'X5540', 'debit', 'by', '1080.0', 'on', 'date', '30mar25', 'trf', 'to', 'MALAPPURAM', 'TRADE', 'Refno', '016207021269', '.', 'if', 'not', 'u', '?', 'call', '1800111109.', '�', '-SBI']


In [ ]:

for chunk in doc.noun_chunks:
    print(chunk.text, chunk.root.text, chunk.root.dep_,
            chunk.root.head.text)


Dear UPI user A/C X5540 X5540 ROOT X5540
date date pobj on
MALAPPURAM TRADE Refno Refno pobj to
u u nsubj call
? ? nsubj call
1800111109.� � dobj call


In [ ]:
from spacy import displacy
displacy.serve(doc, style="ent")


Using the 'ent' visualizer
Serving on http://0.0.0.0:5000 ...



In [3]:
!pip install spacy-lookups-data
!python -m spacy download en


import random
import spacy
from spacy.util import minibatch
from spacy.training.example import Example


⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 105.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
train_data = [
    ("How much for 15 apples?", {"entities": [(13, 15, "QUANTITY"), (16, 22, "PRODUCT")]}),
    ("What's the cost of 2 pineapples?", {"entities": [(21, 22, "QUANTITY"), (23, 33, "PRODUCT")]}),
    ("I need 6 bottles of milk.", {"entities": [(7, 8, "QUANTITY"), (9, 22, "PRODUCT")]}),
    ("Give me 3 dozen eggs.", {"entities": [(8, 9, "QUANTITY"), (10, 21, "PRODUCT")]}),
    ("Price for 10 watermelons?", {"entities": [(11, 13, "QUANTITY"), (14, 25, "PRODUCT")]}),
    ("How much are 5 lemons?", {"entities": [(13, 14, "QUANTITY"), (15, 21, "PRODUCT")]}),
    ("Can I get 7 packets of sugar?", {"entities": [(10, 11, "QUANTITY"), (12, 28, "PRODUCT")]}),
    ("What will 4 mangoes cost?", {"entities": [(10, 11, "QUANTITY"), (12, 19, "PRODUCT")]}),
    ("Order 18 chocolate bars.", {"entities": [(6, 8, "QUANTITY"), (9, 24, "PRODUCT")]}),
    ("Need 9 loaves of bread.", {"entities": [(5, 6, "QUANTITY"), (7, 22, "PRODUCT")]}),
    ("What is the rate of 3 cucumbers?", {"entities": [(22, 23, "QUANTITY"), (24, 33, "PRODUCT")]}),
    ("I want 12 soda cans.", {"entities": [(7, 9, "QUANTITY"), (10, 19, "PRODUCT")]}),
    ("Quote for 20 cheese slices.", {"entities": [(11, 13, "QUANTITY"), (14, 27, "PRODUCT")]}),
    ("How much do 11 lollipops cost?", {"entities": [(13, 15, "QUANTITY"), (16, 25, "PRODUCT")]}),
    ("Give me 8 rolls of tissue.", {"entities": [(8, 9, "QUANTITY"), (10, 25, "PRODUCT")]}),
    ("I'd like 5 cans of beans.", {"entities": [(9, 10, "QUANTITY"), (11, 24, "PRODUCT")]}),
    ("Add 14 packs of noodles.", {"entities": [(4, 6, "QUANTITY"), (7, 22, "PRODUCT")]}),
    ("Get 2 tubs of ice cream.", {"entities": [(4, 5, "QUANTITY"), (6, 24, "PRODUCT")]}),
    ("Need a price for 6 onions.", {"entities": [(17, 18, "QUANTITY"), (19, 25, "PRODUCT")]}),
    ("Buy 30 bottles of juice.", {"entities": [(4, 6, "QUANTITY"), (7, 23, "PRODUCT")]}),
    ("Total cost for 9 chocolate bars?", {"entities": [(15, 16, "QUANTITY"), (17, 32, "PRODUCT")]}),
    ("How much for 4 water bottles?", {"entities": [(13, 14, "QUANTITY"), (15, 30, "PRODUCT")]}),
    ("Pick up 10 liters of oil.", {"entities": [(8, 10, "QUANTITY"), (11, 23, "PRODUCT")]}),
    ("I want to purchase 16 tea bags.", {"entities": [(20, 22, "QUANTITY"), (23, 31, "PRODUCT")]}),
    ("Order for 13 bananas.", {"entities": [(10, 12, "QUANTITY"), (13, 20, "PRODUCT")]}),
    ("Get me 2 cartons of milk.", {"entities": [(7, 8, "QUANTITY"), (9, 24, "PRODUCT")]}),
    ("Please provide cost for 5 oranges.", {"entities": [(24, 25, "QUANTITY"), (26, 33, "PRODUCT")]}),
    ("Cost of 7 bottles of soda?", {"entities": [(8, 9, "QUANTITY"), (10, 25, "PRODUCT")]}),
    ("Do you have 18 ice cream cones?", {"entities": [(13, 15, "QUANTITY"), (16, 34, "PRODUCT")]}),
    ("I'd like to buy 22 cupcakes.", {"entities": [(17, 19, "QUANTITY"), (20, 29, "PRODUCT")]}),
    ("How much are 9 yogurt cups?", {"entities": [(13, 14, "QUANTITY"), (15, 27, "PRODUCT")]}),
    ("Give me 6 apples and 7 pears.", {"entities": [(8, 9, "QUANTITY"), (10, 16, "PRODUCT"), (21, 22, "QUANTITY"), (23, 28, "PRODUCT")]}),
    ("I want 11 cartons of eggs.", {"entities": [(7, 9, "QUANTITY"), (10, 25, "PRODUCT")]}),
    ("How much for 4 bread loaves?", {"entities": [(13, 14, "QUANTITY"), (15, 27, "PRODUCT")]}),
    ("Buy 5 cans of soda please.", {"entities": [(4, 5, "QUANTITY"), (6, 17, "PRODUCT")]}),
    ("Quote price for 3 kilograms of rice.", {"entities": [(17, 18, "QUANTITY"), (19, 37, "PRODUCT")]}),
    ("Add 1 packet of biscuits to the list.", {"entities": [(4, 5, "QUANTITY"), (6, 22, "PRODUCT")]}),
    ("Include 12 energy drinks.", {"entities": [(8, 10, "QUANTITY"), (11, 25, "PRODUCT")]}),
    ("Get me 7 cans of tomato soup.", {"entities": [(7, 8, "QUANTITY"), (9, 27, "PRODUCT")]}),
    ("Buy 19 eggs and 15 onions.", {"entities": [(4, 6, "QUANTITY"), (7, 11, "PRODUCT"), (16, 18, "QUANTITY"), (19, 25, "PRODUCT")]}),
    ("Can I get 5 snack bars?", {"entities": [(10, 11, "QUANTITY"), (12, 22, "PRODUCT")]}),
    ("Pick up 8 soap bars and 3 shampoo bottles.", {"entities": [(8, 9, "QUANTITY"), (10, 20, "PRODUCT"), (25, 26, "QUANTITY"), (27, 43, "PRODUCT")]}),
    ("Give 6 packs of flour.", {"entities": [(5, 6, "QUANTITY"), (7, 20, "PRODUCT")]}),
    ("How much are 2 liters of milk?", {"entities": [(13, 14, "QUANTITY"), (15, 28, "PRODUCT")]}),
    ("I want 17 bags of chips.", {"entities": [(7, 9, "QUANTITY"), (10, 23, "PRODUCT")]}),
    ("Price for 3 bottles of honey?", {"entities": [(11, 12, "QUANTITY"), (13, 28, "PRODUCT")]}),
    ("Cost for 21 chocolate cookies?", {"entities": [(10, 12, "QUANTITY"), (13, 31, "PRODUCT")]}),
    ("Add 2 boxes of cereal.", {"entities": [(4, 5, "QUANTITY"), (6, 20, "PRODUCT")]}),
    ("Give me 10 candy bars.", {"entities": [(8, 10, "QUANTITY"), (11, 22, "PRODUCT")]}),
]


In [5]:
nlp = spacy.load('en_core_web_sm')

if 'ner' not in nlp.pipe_names:
   ner = nlp.add_pipe('ner')
else:
   ner = nlp.get_pipe('ner')

for _, annotations in train_data:
  for ent in annotations['entities']:
    if ent[2] not in ner.labels:
      ner.add_label(ent[2])

other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']
with nlp.disable_pipes(*other_pipes):
  optimizer = nlp.begin_training()

  epochs = 50
  for epoch in range(epochs):
    random.shuffle(train_data)
    losses = {}
    batches = minibatch(train_data, size=2)
    for batch in batches:
      examples = []
      for text, annotations in batch:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        examples.append(example)
      nlp.update(examples, drop=0.5, losses=losses)
    print(f"Epoch {epoch + 1}/{epochs} - Losses: {losses}")

nlp.to_disk('custom_ner_model')
trained_nlp = spacy.load('custom_ner_model')

test_suits = [
    "Hpow much for 15 apples?",
    "What's the cost of 2 pineapples?",
    "I need 6 bottles of milk.",
    "Give me 3 dozen eggs.",
    "Price for 10 watermelons?",
]

for test_suit in test_suits:
    doc = trained_nlp(test_suit)
    print(f"Test Suit: {test_suit}")
    for ent in doc.ents:
        print(f"{ent.label_}: {ent.text}")

/usr/local/lib/python3.11/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Pick up 8 soap bars and 3 shampoo bottles." with entities "[(8, 9, 'QUANTITY'), (10, 20, 'PRODUCT'), (25, 26,...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Give 6 packs of flour." with entities "[(5, 6, 'QUANTITY'), (7, 20, 'PRODUCT')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Buy 5 cans of soda please." with e

Epoch 1/50 - Losses: {'ner': np.float32(210.25128)}
Epoch 2/50 - Losses: {'ner': np.float32(112.50442)}
Epoch 3/50 - Losses: {'ner': np.float32(94.16381)}
Epoch 4/50 - Losses: {'ner': np.float32(79.821266)}
Epoch 5/50 - Losses: {'ner': np.float32(62.915348)}
Epoch 6/50 - Losses: {'ner': np.float32(43.96737)}
Epoch 7/50 - Losses: {'ner': np.float32(35.667988)}
Epoch 8/50 - Losses: {'ner': np.float32(43.791344)}
Epoch 9/50 - Losses: {'ner': np.float32(32.22931)}
Epoch 10/50 - Losses: {'ner': np.float32(16.615156)}
Epoch 11/50 - Losses: {'ner': np.float32(21.411682)}
Epoch 12/50 - Losses: {'ner': np.float32(18.286745)}
Epoch 13/50 - Losses: {'ner': np.float32(18.994282)}
Epoch 14/50 - Losses: {'ner': np.float32(8.152746)}
Epoch 15/50 - Losses: {'ner': np.float32(10.866959)}
Epoch 16/50 - Losses: {'ner': np.float32(11.885245)}
Epoch 17/50 - Losses: {'ner': np.float32(11.777287)}
Epoch 18/50 - Losses: {'ner': np.float32(8.588908)}
Epoch 19/50 - Losses: {'ner': np.float32(8.129182)}
Epoch 20